In [1]:

#STEP 1: Data Loading, Feature Integration, and Preprocessing
# Data.tsv is stored locally in the 
# same directory as of this python file
import pandas as pd
df = pd.read_csv('trainingSet.tsv',sep = '\t') 
display(df)

,bond,id,label_mutagenic
0,http://dl-learner.org/carcinogenesis#d187,3.0,1.0
1,http://dl-learner.org/carcinogenesis#d133,4.0,0.0
2,http://dl-learner.org/carcinogenesis#d23_2,6.0,0.0
3,http://dl-learner.org/carcinogenesis#d312,8.0,0.0
4,http://dl-learner.org/carcinogenesis#d81,10.0,0.0
...,...,...,...
267,http://dl-learner.org/carcinogenesis#d145,336.0,1.0
268,http://dl-learner.org/carcinogenesis#d94,337.0,0.0
269,http://dl-learner.org/carcinogenesis#d175,338.0,0.0
270,http://dl-learner.org/carcinogenesis#d34,339.0,0.0


In [2]:
# STEP 2:Install Required Dependencies for RDF Processing

!pip install rdflib

In [3]:
import os
import pandas as pd
from rdflib import Graph, Namespace, RDF, URIRef
from collections import Counter

INPUT_FILE = "mutag_stripped.nt"
OUTPUT_FILE = "mutag_compound_features.csv"

CARC = Namespace("http://dl-learner.org/carcinogenesis#")

def local_name(x):
    return str(x).split("#")[-1].split("/")[-1]

# Load RDF
g = Graph()
g.parse(INPUT_FILE, format="nt")

print("Triples loaded:", len(g), flush=True)

# Step 3: get real molecules/compounds only
compounds = sorted(set(g.subjects(RDF.type, CARC.Compound)), key=str)

print("Number of compounds:", len(compounds), flush=True)

rows = []

# Step 4: build one feature row per compound
for i, compound in enumerate(compounds):
    row = {
        "molecule": local_name(compound)
    }

    features = Counter()

    # Atom features
    atoms = list(g.objects(compound, CARC.hasAtom))
    row["num_atoms"] = len(atoms)

    for atom in atoms:
        for atom_type in g.objects(atom, RDF.type):
            features[f"atom_type__{local_name(atom_type)}"] += 1

        for charge in g.objects(atom, CARC.charge):
            features[f"charge__{local_name(charge)}"] += 1

    # Bond features
    bonds = list(g.objects(compound, CARC.hasBond))
    row["num_bonds"] = len(bonds)

    for bond in bonds:
        for bond_type in g.objects(bond, RDF.type):
            features[f"bond_type__{local_name(bond_type)}"] += 1

    # Structure features
    structures = list(g.objects(compound, CARC.hasStructure))
    row["num_structures"] = len(structures)

    for structure in structures:
        for structure_type in g.objects(structure, RDF.type):
            features[f"structure_type__{local_name(structure_type)}"] += 1

    # Compound-level test/label features
    for p, o in g.predicate_objects(compound):
        pred = local_name(p)

        if pred in ["hasAtom", "hasBond", "hasStructure", "type"]:
            continue

        obj = local_name(o)
        row[f"{pred}"] = obj

    # Add counted features
    row.update(features)

    rows.append(row)

    if i % 25 == 0:
        print("Processed compounds:", i, flush=True)

# Step 5: convert to DataFrame
df = pd.DataFrame(rows).fillna(0)

# Step 6: convert numeric feature columns to small integers
for col in df.columns:
    if col != "molecule":
        if col.startswith(("atom_type__", "bond_type__", "structure_type__", "charge__")):
            df[col] = df[col].astype("uint8")

print("Final shape:", df.shape, flush=True)
print(df.head(), flush=True)

# Step 7: save only small compound-level matrix
df.to_csv(OUTPUT_FILE, index=False)

print("Saved:", OUTPUT_FILE, flush=True)
print("File size MB:", os.path.getsize(OUTPUT_FILE) / (1024 ** 2))

Triples loaded: 74227


Number of compounds: 340


Processed compounds: 0


Processed compounds: 25


Processed compounds: 50


...

Final shape:

 (340, 1227)


  molecule  num_atoms  num_bonds  num_structures salmonella amesTestPositive  \
0       d1         26         28              11       true             true   
1      d10         16         16               5       true             true   
2     d100         26         27               8      false            false   
3     d101         22         24              23      false            false   
4     d102          8          7              11      false            false   

  salmonella_n cytogen_sce cytogen_ca  atom_type__Carbon-22  ...  \
0         true        true       true                    12  ...   
1         true        true       true                     6  ...   
2            0        true      false                    12  ...   
3            0        true      false                     0  ...   
4            0        true      false                     0  ...   

   charge__-0.207  charge__0.393  charge__0.611  charge__0.474  charge__0.434  \
0               0            

Saved: mutag_compound_features.csv


File size MB: 0.8211565017700195


In [5]:

# STEP 8: Merge features with train/test


import pandas as pd
import numpy as np

TRAIN_FILE = "trainingSet.tsv"
TEST_FILE = "testSet.tsv"
FEATURE_FILE = "mutag_compound_features.csv"

train_df = pd.read_csv(TRAIN_FILE, sep="\t")
test_df = pd.read_csv(TEST_FILE, sep="\t")
features_df = pd.read_csv(FEATURE_FILE)

# Make sure feature ID column is called bond
if "molecule" in features_df.columns:
    features_df = features_df.rename(columns={"molecule": "bond"})

# Extract local molecule ID from URI
# Make sure feature ID column is called bond
if "molecule" in features_df.columns:
    features_df = features_df.rename(columns={"molecule": "bond"})

# Extract local molecule ID from URI
def extract_id(x):
    x = str(x).strip()
    if "#" in x:
        return x.split("#")[-1]
    return x

train_df["bond"] = train_df["bond"].apply(extract_id)
test_df["bond"] = test_df["bond"].apply(extract_id)
features_df["bond"] = features_df["bond"].astype(str).str.strip()

# Merge
Xtrain = train_df.merge(features_df, on="bond", how="left")
Xtest = test_df.merge(features_df, on="bond", how="left")

print("Matched train rows:", Xtrain["num_atoms"].notna().sum(), "out of", len(Xtrain))
print("Matched test rows:", Xtest["num_atoms"].notna().sum(), "out of", len(Xtest))

if Xtrain["num_atoms"].notna().sum() == 0:
    raise ValueError("Merge still failed. Check ID columns.")

# Labels
ytrain = Xtrain["label_mutagenic"]
ytest = Xtest["label_mutagenic"]

# Drop non-feature columns
drop_cols = ["id", "bond", "label_mutagenic"]

Xtrain_features = Xtrain.drop(columns=drop_cols, errors="ignore")
Xtest_features = Xtest.drop(columns=drop_cols, errors="ignore")

# Keep only numeric columns; remove string columns
Xtrain_features = Xtrain_features.select_dtypes(include=[np.number])
Xtest_features = Xtest_features.select_dtypes(include=[np.number])

# Fill missing values
Xtrain_features = Xtrain_features.fillna(0)
Xtest_features = Xtest_features.fillna(0)

# Align train/test columns
Xtrain_features, Xtest_features = Xtrain_features.align(
    Xtest_features,
    join="left",
    axis=1,
    fill_value=0
)

print("Xtrain_features shape:", Xtrain_features.shape)
print("Xtest_features shape:", Xtest_features.shape)
print("Training labels:")
print(ytrain.value_counts())

print("Top feature variances:")
print(Xtrain_features.var().sort_values(ascending=False).head(10))

Matched train rows: 272 out of 272
Matched test rows: 68 out of 68
Xtrain_features shape: (272, 1213)
Xtest_features shape: (68, 1213)
Training labels:
label_mutagenic
0.0    166
1.0    106
Name: count, dtype: int64
Top feature variances:
num_bonds                481.366711
num_atoms                458.475567
bond_type__Bond-1        413.228769
atom_type__Hydrogen-3    146.702938
num_structures            53.643247
atom_type__Carbon-10      37.845005
bond_type__Bond-7         35.470141
atom_type__Carbon-22      26.641727
charge__0.069             21.612112
charge__0.05              12.813762
dtype: float64


In [6]:

# STEP 9: Train Logistic Regression Model

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

lr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=5000,
        random_state=42,
        solver="liblinear"
    ))
])
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
import numpy as np

# Initialize the Linear Regression model
#lr_model = LogisticRegression()

# Train the model
lr_model.fit(Xtrain_features, ytrain)

# Predict continuous values
lr_predictions = lr_model.predict(Xtest_features)

# Convert continuous predictions to binary classes
lr_predictions = np.where(lr_predictions >= 0.5, 1, 0)

# Evaluate the model
lr_accuracy = accuracy_score(ytest, lr_predictions)

print("Logistic Regression Accuracy:", lr_accuracy)

print("\nClassification Report:")
print(classification_report(ytest, lr_predictions, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(ytest, lr_predictions))

Logistic Regression Accuracy: 0.6470588235294118

Classification Report:
              precision    recall  f1-score   support

         0.0       0.71      0.80      0.75        45
         1.0       0.47      0.35      0.40        23

    accuracy                           0.65        68
   macro avg       0.59      0.57      0.57        68
weighted avg       0.63      0.65      0.63        68


Confusion Matrix:
[[36  9]
 [15  8]]


In [7]:
# STEP 10:Train SVM model

from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Initialize the SVM classifier
svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LinearSVC(
        random_state=42,
        max_iter=10000
    ))
])

# Train the model
svm_model.fit(Xtrain_features, ytrain)

# Make predictions
svm_predictions = svm_model.predict(Xtest_features)

# Evaluate the model
accuracy = accuracy_score(ytest, svm_predictions)

print("Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(ytest, svm_predictions))

print("\nConfusion Matrix:")
print(confusion_matrix(ytest, svm_predictions))

Accuracy: 0.5588235294117647

Classification Report:
              precision    recall  f1-score   support

         0.0       0.66      0.69      0.67        45
         1.0       0.33      0.30      0.32        23

    accuracy                           0.56        68
   macro avg       0.50      0.50      0.50        68
weighted avg       0.55      0.56      0.55        68


Confusion Matrix:
[[31 14]
 [16  7]]


In [8]:
# STEP 11: Train Decision Tree model



from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Initialize Decision Tree classifier
dt_model = DecisionTreeClassifier(
    random_state=42,
    max_depth=5
)

# Train the model
dt_model.fit(Xtrain_features, ytrain)

# Make predictions
dt_predictions = dt_model.predict(Xtest_features)

# Evaluate the model
dt_accuracy = accuracy_score(ytest, dt_predictions)

print("Decision Tree Accuracy:", dt_accuracy)

print("\nClassification Report:")
print(classification_report(ytest, dt_predictions))

print("\nConfusion Matrix:")
print(confusion_matrix(ytest, dt_predictions))

Decision Tree Accuracy: 0.7647058823529411

Classification Report:
              precision    recall  f1-score   support

         0.0       0.76      0.93      0.84        45
         1.0       0.77      0.43      0.56        23

    accuracy                           0.76        68
   macro avg       0.77      0.68      0.70        68
weighted avg       0.77      0.76      0.74        68


Confusion Matrix:
[[42  3]
 [13 10]]


In [18]:
#STEP 12: Generation of Model Explanation Figures

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

feature_names = Xtrain_features.columns

# Logistic Regression
log_coef = pd.Series(
    lr_model.named_steps["classifier"].coef_.ravel(),
    index=feature_names
)

log_coef = log_coef.reindex(log_coef.abs().sort_values(ascending=False).index)

plt.figure(figsize=(10,6))
log_coef.head(15).sort_values().plot(kind="barh")
plt.title("Top 15 Logistic Regression Coefficients")
plt.tight_layout()
# Save figure
plt.savefig("logistic_regression_coefficients.png",
            dpi=300,
            bbox_inches="tight")

plt.show()


# Linear SVM
svm_coef = pd.Series(
    svm_model.named_steps["classifier"].coef_.ravel(),
    index=feature_names
)

svm_coef = svm_coef.reindex(svm_coef.abs().sort_values(ascending=False).index)

plt.figure(figsize=(10,6))
svm_coef.head(15).sort_values().plot(kind="barh")
plt.title("Top 15 Linear SVM Feature Weights")
plt.tight_layout()
plt.savefig("linear_svm_feature_weights.png",
            dpi=300,
            bbox_inches="tight")

plt.show()


# Decision Tree Feature Importance
importance = pd.Series(
    dt_model.feature_importances_,
    index=feature_names
).sort_values(ascending=False)

plt.figure(figsize=(10,6))
importance.head(15).sort_values().plot(kind="barh")
plt.title("Top 15 Decision Tree Feature Importances")
plt.tight_layout()
plt.savefig("decision_tree_feature_importance.png",
            dpi=300,
            bbox_inches="tight")

plt.show()

<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

In [10]:

# STEP 13: Quantitative Evaluation of Explanations

import pandas as pd
import numpy as np
import time

def explanation_metrics(model_name, method_name, values, explanation_time):
    values = np.abs(np.ravel(values))
    mean_val = np.mean(values)

    important_features = np.sum(values > mean_val)

    return {
        "Model": model_name,
        "Explanation Method": method_name,
        "Mean Importance": mean_val,
        "Max Importance": np.max(values),
        "No. of Important Features": important_features,
        "Sparsity": 1 - (important_features / len(values)),
        "Explanation Time (s)": explanation_time
    }


results = []

# Logistic Regression
start = time.time()
log_values = lr_model.named_steps["classifier"].coef_.ravel()
end = time.time()

results.append(
    explanation_metrics(
        "Logistic Regression",
        "Coefficient analysis",
        log_values,
        end - start
    )
)

# Linear SVM
start = time.time()
svm_values = svm_model.named_steps["classifier"].coef_.ravel()
end = time.time()

results.append(
    explanation_metrics(
        "Linear SVM",
        "Feature weight analysis",
        svm_values,
        end - start
    )
)

# Decision Tree
start = time.time()
dt_values = dt_model.feature_importances_
end = time.time()

results.append(
    explanation_metrics(
        "Decision Tree",
        "Gini-based feature importance",
        dt_values,
        end - start
    )
)

# Create table
explanation_eval_table = pd.DataFrame(results).round(4)

print("Quantitative Evaluation of Model Explanations")
display(explanation_eval_table)

# Save files for report
explanation_eval_table.to_csv(
    "quantitative_explanation_evaluation.csv",
    index=False
)

explanation_eval_table.to_latex(
    "quantitative_explanation_evaluation.tex",
    index=False,
    caption="Quantitative evaluation of explanations for Logistic Regression, Linear SVM, and Decision Tree.",
    label="tab:explanation_eval"
)

Quantitative Evaluation of Model Explanations


,Model,Explanation Method,Mean Importance,Max Importance,No. of Important Features,Sparsity,Explanation Time (s)
0,Logistic Regression,Coefficient analysis,0.0585,0.8246,468,0.6142,0.0001
1,Linear SVM,Feature weight analysis,0.0197,0.7692,369,0.6958,0.0001
2,Decision Tree,Gini-based feature importance,0.0008,0.4823,10,0.9918,0.0001


In [17]:
#STEP 14: Compare Model Performance

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

def get_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "F1 Score": f1_score(y_true, y_pred, average="weighted", zero_division=0)
    }

results = []

models = {
    "Logistic Regression": lr_model,
    "SVM": svm_model,
    "Decision Tree": dt_model
}

for model_name, model in models.items():

    # Predictions on training set
    train_pred = model.predict(Xtrain_features)

    # Predictions on test set
    test_pred = model.predict(Xtest_features)

    # Compute metrics
    train_metrics = get_metrics(ytrain, train_pred)
    test_metrics = get_metrics(ytest, test_pred)

    results.append({
        "Model": model_name,
        "Train Accuracy": train_metrics["Accuracy"],
        "Test Accuracy": test_metrics["Accuracy"],
        "Train Precision": train_metrics["Precision"],
        "Test Precision": test_metrics["Precision"],
        "Train Recall": train_metrics["Recall"],
        "Test Recall": test_metrics["Recall"],
        "Train F1": train_metrics["F1 Score"],
        "Test F1": test_metrics["F1 Score"]
    })

# Create DataFrame
results_df = pd.DataFrame(results)

print(results_df)

results_df = results_df.round(4)

latex_table = results_df.to_latex(
    index=False,
    caption="Training and test performance of the evaluated models.",
    label="tab:model_performance",
    float_format="%.4f"
)

with open("model_train_test__performance.tex", "w") as f:
    f.write(latex_table)

print("LaTeX table saved as model_performance.tex")

# Save to CSV
results_df.to_csv("model_train_test_performance.csv", index=False)

print("\nResults saved as 'model_train_test_performance.csv'")

                 Model  Train Accuracy  Test Accuracy  Train Precision  \
0  Logistic Regression        0.992647       0.647059         0.992647   
1                  SVM        0.996324       0.558824         0.996346   
2        Decision Tree        0.878676       0.764706         0.881597   

   Test Precision  Train Recall  Test Recall  Train F1   Test F1  
0        0.626298      0.992647     0.647059  0.992647  0.631618  
1        0.549228      0.996324     0.558824  0.996320  0.553592  
2        0.765529      0.878676     0.764706  0.876343  0.743791  
LaTeX table saved as model_performance.tex

Results saved as 'model_train_test_performance.csv'


In [12]:
# STEP 15: Classification reports



from sklearn.metrics import classification_report
import numpy as np

# Get class names from the test labels
class_names = [str(c) for c in sorted(np.unique(ytest))]

# Linear Regression Classification Report
print("\n========== Linear Regression Classification Report ==========\n")
print(
    classification_report(
        ytest,
        lr_predictions,
        target_names=class_names,
        zero_division=0
    )
)

# SVM Classification Report
print("\n========== SVM Classification Report ==========\n")
print(
    classification_report(
        ytest,
        svm_predictions,
        target_names=class_names,
        zero_division=0
    )
)

# Decision Tree Classification Report
print("\n========== Decision Tree Classification Report ==========\n")
print(
    classification_report(
        ytest,
        dt_predictions,
        target_names=class_names,
        zero_division=0
    )
)


========== Linear Regression Classification Report ==========

              precision    recall  f1-score   support

         0.0       0.71      0.80      0.75        45
         1.0       0.47      0.35      0.40        23

    accuracy                           0.65        68
   macro avg       0.59      0.57      0.57        68
weighted avg       0.63      0.65      0.63        68


========== SVM Classification Report ==========

              precision    recall  f1-score   support

         0.0       0.66      0.69      0.67        45
         1.0       0.33      0.30      0.32        23

    accuracy                           0.56        68
   macro avg       0.50      0.50      0.50        68
weighted avg       0.55      0.56      0.55        68


========== Decision Tree Classification Report ==========

              precision    recall  f1-score   support

         0.0       0.76      0.93      0.84        45
         1.0       0.77      0.43      0.56        23

    accu

In [ ]:
# STEP 16: Confusion matrix 


from sklearn.metrics import confusion_matrix

print("Linear Regression Confusion Matrix:")
print(confusion_matrix(ytest, lr_predictions))

print("SVM Confusion Matrix:")
print(confusion_matrix(ytest, svm_predictions))

print("\nDecision Tree Confusion Matrix:")
print(confusion_matrix(ytest, dt_predictions))

In [16]:
# STEP 17: Save outputs


import os

# Save feature matrix
feature_file = "mutag_compound_features.csv"
features_df.to_csv(feature_file, index=False)

# Save model performance comparison
if 'results_df' in locals():

    # Round values for better readability
    results_df = results_df.round(4)

    # Save CSV (optional)
    performance_file = "model_performance_comparison.csv"
    results_df.to_csv(performance_file, index=False)

    # Save as LaTeX table
    latex_file = "model_performance_comparison.tex"

    with open(latex_file, "w") as f:
        f.write(results_df.to_latex(
            index=False,
            caption="Training and test performance of the evaluated machine learning models.",
            label="tab:model_performance",
            float_format="%.4f",
            escape=False
        ))

    print(f"CSV saved as '{performance_file}'")
    print(f"LaTeX table saved as '{latex_file}'")
   
    print("\nFiles saved successfully:")
    print(f"1. {feature_file}")
    print(f"2. {performance_file}")

    print(f"\nFeature Matrix Size: {os.path.getsize(feature_file)/(1024**2):.2f} MB")
    print(f"Performance File Size: {os.path.getsize(performance_file)/(1024**2):.2f} MB")

else:
    print("\nFeature matrix saved successfully.")
    print(f"File: {feature_file}")
    print("\nresults_df not found. Please run the model evaluation step before saving the performance comparison.")

CSV saved as 'model_performance_comparison.csv'
LaTeX table saved as 'model_performance_comparison.tex'

Files saved successfully:
1. mutag_compound_features.csv
2. model_performance_comparison.csv

Feature Matrix Size: 0.82 MB
Performance File Size: 0.00 MB
